In [13]:
import numpy as np
import pandas as pd
from sklearn import ensemble
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [14]:
df = pd.read_csv('/content/banana_quality.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Size         8000 non-null   float64
 1   Weight       8000 non-null   float64
 2   Sweetness    8000 non-null   float64
 3   Softness     8000 non-null   float64
 4   HarvestTime  8000 non-null   float64
 5   Ripeness     8000 non-null   float64
 6   Acidity      8000 non-null   float64
 7   Quality      8000 non-null   object 
dtypes: float64(7), object(1)
memory usage: 500.1+ KB


In [15]:
df.head()

,Size,Weight,Sweetness,Softness,HarvestTime,Ripeness,Acidity,Quality
0,-1.924968,0.468078,3.077832,-1.472177,0.294799,2.435570,0.271290,Good
1,-2.409751,0.486870,0.346921,-2.495099,-0.892213,2.067549,0.307325,Good
2,-0.357607,1.483176,1.568452,-2.645145,-0.647267,3.090643,1.427322,Good
3,-0.868524,1.566201,1.889605,-1.273761,-1.006278,1.873001,0.477862,Good
4,0.651825,1.319199,-0.022459,-1.209709,-1.430692,1.078345,2.812442,Good


Istrenirajte ansambl modele AdaBoost i RandomForest. Standardizacija u ovom slučaju nije potrebna jer ako se ne zada bazni model, koristi se stablo odlučivanja koje ne zahtijeva standardizaciju. Koristite GridSearcCV, a mijenjajte samo jedan hiperparametar po želji (dvije vrijednosti) i ispišite točnosti.

In [18]:
#target
y = df["Quality"]
X = df.drop(columns=['Quality'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


pipe_ada = Pipeline([
    ("scaler", StandardScaler()),
    ("ada", AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=150,  learning_rate=0.5, random_state=42))
])

param_grid_ada = {
    'ada__n_estimators': [100, 150]}


grid_ada = GridSearchCV(pipe_ada,param_grid=param_grid_ada,cv=5,scoring='accuracy',return_train_score=True)
grid_ada.fit(X_train, y_train)

print("Najbolji parametri za ADA:", grid_ada.best_params_)
print(f"Najbolja točnost (CV): {grid_ada.best_score_ * 100:.2f}%")

y_pred_log = grid_ada.predict(X_test)
print(f"Točnost na test skupu: {accuracy_score(y_test, y_pred_log) * 100:.2f}%")


print("------------------------------------------------------------")


pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42))
])


param_grid_rf = {
    'rf__max_depth': [None, 5]
}

grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='accuracy', return_train_score=True)
grid_rf.fit(X_train, y_train)

print("Najbolji parametri za RF:", grid_rf.best_params_)
print(f"Najbolja točnost (CV): {grid_rf.best_score_ * 100:.2f}%")

y_pred_log = grid_rf.predict(X_test)
print(f"Točnost na test skupu: {accuracy_score(y_test, y_pred_log) * 100:.2f}%")


Najbolji parametri za ADA: {'ada__n_estimators': 150}
Najbolja točnost (CV): 89.64%
Točnost na test skupu: 88.31%
------------------------------------------------------------
Najbolji parametri za RF: {'rf__max_depth': None}
Najbolja točnost (CV): 97.08%
Točnost na test skupu: 96.62%
